<div style="width: 720px; background: linear-gradient(135deg, #0f0c29, #302b63, #24243e); padding: 36px 40px 28px 40px; border-radius: 12px; color: white; font-family: 'Segoe UI', Arial, sans-serif;">

  <div style="border-left: 4px solid #e94560; padding-left: 16px; margin-bottom: 24px;">
    <div style="font-size: 0.75em; color: #e94560; letter-spacing: 2px; text-transform: uppercase; margin-bottom: 6px;">Trabalho Final</div>
    <div style="font-size: 1.55em; font-weight: 700; color: #ffffff; line-height: 1.3; margin-bottom: 6px;">🧬 Algoritmo Genético para o Problema de<br/>Job-Shop Scheduling Flexível</div>
    <div style="font-size: 0.88em; color: #a8b2d8; font-style: italic;">Flexible Job-Shop Scheduling Problem (FJSP) — Instância Estática</div>
  </div>

  <table style="width: 100%; border-collapse: collapse; font-size: 0.88em;">
    <tr>
      <td style="width: 50%; padding: 10px 16px; border-top: 1px solid rgba(255,255,255,0.08);">
        <span style="display: block; font-size: 0.7em; color: #e94560; text-transform: uppercase; letter-spacing: 1.5px; margin-bottom: 3px;">Aluno</span>
        <span style="color: #e2e8f0; font-weight: 600;">Paulo Ricardo Dalsoto</span>
      </td>
      <td style="width: 50%; padding: 10px 16px; border-top: 1px solid rgba(255,255,255,0.08);">
        <span style="display: block; font-size: 0.7em; color: #e94560; text-transform: uppercase; letter-spacing: 1.5px; margin-bottom: 3px;">Disciplina</span>
        <span style="color: #e2e8f0;">Inteligência Artificial e Aprendizado de Máquina</span>
      </td>
    </tr>
    <tr>
      <td style="padding: 10px 16px; border-top: 1px solid rgba(255,255,255,0.08);">
        <span style="display: block; font-size: 0.7em; color: #e94560; text-transform: uppercase; letter-spacing: 1.5px; margin-bottom: 3px;">Instituição</span>
        <span style="color: #e2e8f0;">Universidade do Vale do Rio dos Sinos — UNISINOS</span>
      </td>
      <td style="padding: 10px 16px; border-top: 1px solid rgba(255,255,255,0.08);">
        <span style="display: block; font-size: 0.7em; color: #e94560; text-transform: uppercase; letter-spacing: 1.5px; margin-bottom: 3px;">Semestre</span>
        <span style="color: #e2e8f0;">2025/2</span>
      </td>
    </tr>
  </table>

</div>

---
## 1. Introdução

O **Problema de Job-Shop Scheduling Flexível** (*Flexible Job-Shop Scheduling Problem*, FJSP) é um dos problemas de otimização combinatória de maior relevância prática e teórica nas áreas de pesquisa operacional e inteligência artificial. Ele modela situações recorrentes em ambientes de manufatura modernos, onde um conjunto de tarefas de produção (*jobs*) precisa ser executado em um conjunto de máquinas de forma eficiente, respeitando restrições tecnológicas e operacionais.

Diferentemente do Job-Shop Scheduling clássico (JSSP), em que cada operação está rigidamente vinculada a uma única máquina, o FJSP incorpora **flexibilidade de roteamento**: cada operação pode ser processada por qualquer máquina de um subconjunto de máquinas elegíveis, cada uma com seu próprio tempo de processamento. Essa característica reflete com maior fidelidade os ambientes industriais reais, como células de manufatura flexível, centros de usinagem CNC e linhas de montagem reconfiguráveis. Por outro lado, ela torna o problema consideravelmente mais complexo, pois o algoritmo de resolução deve tomar decisões tanto de **atribuição** (em qual máquina executar cada operação) quanto de **sequenciamento** (em qual ordem as operações serão processadas em cada máquina).

O FJSP é classificado como **NP-difícil**, o que significa que não se conhece algoritmo de tempo polinomial capaz de resolvê-lo de forma ótima para instâncias de tamanho arbitrário. Por essa razão, ao longo das últimas décadas, a literatura da área desenvolveu uma ampla variedade de abordagens heurísticas e metaheurísticas, entre elas *Simulated Annealing*, Busca Tabu, Otimização por Colônia de Formigas (ACO), Otimização por Enxame de Partículas (PSO) e, em especial, os **Algoritmos Genéticos (AG)**, que têm apresentado resultados competitivos frente a outros métodos.

Neste trabalho, implementamos um Algoritmo Genético para resolver o FJSP na sua variante **estática**, em que todos os jobs e suas operações são conhecidos de antemão, antes do início do escalonamento. Essa variante se opõe ao FJSP dinâmico, no qual novos jobs chegam ao sistema ao longo do tempo. A implementação é inspirada no artigo de referência [[1]](#9-referências), que propõe e avalia um AG com codificação cromossômica dupla especialmente projetada para o FJSP, utilizando as instâncias clássicas de Brandimarte como benchmark.

O objetivo central do trabalho é minimizar o **makespan**, ou seja, o instante de conclusão do último job processado. Essa é a métrica mais amplamente adotada na literatura de scheduling e serve como indicador direto da eficiência do escalonamento obtido.

---
## 2. Formulação do FJSP Estático

### 2.1 Definição e Notação

Para entender o FJSP, é útil começar por um exemplo concreto antes de introduzir a notação formal.

Imagine uma pequena fábrica com **3 jobs** a serem produzidos e **3 máquinas** disponíveis. Cada job representa um produto diferente e precisa passar por uma série de etapas de fabricação chamadas **operações**. A ordem dessas etapas é tecnologicamente obrigatória: não é possível montar uma peça antes de usiná-la, por exemplo. No FJSP, o que diferencia esse cenário do escalonamento clássico é que cada operação pode ser realizada em mais de uma máquina, com tempos de processamento diferentes dependendo da máquina escolhida.

A figura a seguir, extraída do artigo de referência [[1]](#9-referências), ilustra exatamente esse cenário:

<img src="imgs/paper_figure1.png" width="900"/>

*Fonte: extraída do artigo de referência [[1]](#9-referências)*


<!-- ![Figura 1 do Artigo de Referência](imgs/paper_figure1.png) -->
<!-- Figura 1: Exemplo de instância FJSP com jobs, operações e máquinas elegíveis -->

Nessa figura, cada linha colorida representa um job com suas operações. Para cada operação, são listadas as máquinas que podem executá-la junto com o respectivo tempo de processamento. O algoritmo precisa decidir, para cada operação, qual máquina será usada e em que momento ela será iniciada, sempre respeitando a ordem das operações dentro de cada job e garantindo que nenhuma máquina processe duas operações ao mesmo tempo.

**Notação utilizada no restante do trabalho:**

Formalizando o que foi descrito acima: o problema envolve um conjunto de $n$ jobs processados em $m$ máquinas. Cada job $J_i$ é composto por uma sequência de $n_i$ operações que devem ser executadas nessa ordem. Para cada operação $O_{i,j}$ (a $j$-ésima operação do job $i$), existe um subconjunto $M_{i,j}$ de máquinas elegíveis; caso a operação seja atribuída à máquina $k$, ela requer $p_{i,j,k}$ unidades de tempo para ser concluída. Todos esses dados são conhecidos de antemão, o que define a variante **estática** do problema.

O algoritmo precisa determinar duas coisas para cada operação: em qual máquina ela será executada (decisão de **atribuição**) e em que instante ela será iniciada (decisão de **escalonamento**). Uma solução válida é qualquer combinação dessas decisões que respeite todas as restrições descritas na seção seguinte.

### 2.2 Restrições

Toda solução válida para o FJSP precisa respeitar quatro tipos de restrição. A seguir, cada uma é apresentada com uma breve explicação e um exemplo baseado nas instâncias de Brandimarte.

**R1 - Atribuição única**

Cada operação deve ser atribuída a **exatamente uma** das máquinas do seu conjunto elegível. O algoritmo não pode deixar uma operação sem máquina, nem dividi-la entre duas máquinas.

*Exemplo:* $O_{1,1}$ do Job 1 pode ir a $M_1$ (tempo=5) ou $M_3$ (tempo=4). O algoritmo escolhe uma delas. Uma solução que não faça essa escolha, ou que atribua $O_{1,1}$ a $M_2$ (que não é elegível), é inválida.

$$a_{i,j} \in M_{i,j}, \quad \forall\, i, j$$

**R2 - Precedência (ordem tecnológica)**

As operações de um job precisam ser executadas **na ordem em que aparecem**. A próxima operação só pode começar depois que a anterior terminar completamente.

*Exemplo:* No Job 1, a operação $O_{1,2}$ não pode começar antes de $O_{1,1}$ ser concluída. Se $O_{1,1}$ começa no instante 0 e dura 4 unidades de tempo (caso seja alocada em $M_3$), então $O_{1,2}$ só pode começar no instante 4 ou depois.

$$S_{i,j} + p_{i,j,a_{i,j}} \leq S_{i,j+1}$$

**R3 - Capacidade de máquina (sem sobreposição)**

Cada máquina só pode processar **uma operação por vez**. Duas operações atribuídas à mesma máquina não podem se sobrepor no tempo.

*Exemplo:* Suponha que $O_{1,3}$ e $O_{2,1}$ (de jobs diferentes) sejam ambas atribuídas a $M_3$. Se $O_{1,3}$ começa no instante 5 e dura 4 unidades, então $O_{2,1}$ só pode começar no instante 9 ou mais tarde — ou terminar antes do instante 5. Não existe a opção de ambas rodarem ao mesmo tempo.

$$S_{i,j} + p_{i,j,k} \leq S_{i',j'} \quad \text{ou} \quad S_{i',j'} + p_{i',j',k} \leq S_{i,j}$$

**R4 - Não-preempção**

Uma vez que uma operação começa a ser processada, ela não pode ser pausada ou interrompida para dar lugar a outra. O processamento é sempre contínuo do início ao fim.

*Exemplo:* Se $O_{1,4}$ foi atribuída a $M_1$ com tempo de processamento igual a 1, ela ocupa $M_1$ por exatamente 1 unidade de tempo, sem interrupção.

### 2.3 Função Objetivo

O objetivo é minimizar o **makespan** $C_{\max}$, definido como o instante de conclusão da última operação entre todos os jobs:

$$\text{Minimizar} \quad C_{\max} = \max_{i=1}^{n} C_i = \max_{i=1}^{n} \left( S_{i,n_i} + p_{i,n_i,\, a_{i,n_i}} \right)$$

onde $C_i = S_{i,n_i} + p_{i,n_i,\, a_{i,n_i}}$ é o tempo de conclusão do job $J_i$.

O makespan representa o tempo total de ocupação do sistema de produção, sendo um indicador direto da eficiência do escalonamento.

### 2.4 Complexidade e Espaço de Busca

O FJSP é **NP-difícil** mesmo em versões simplificadas. Isso decorre da combinação de dois subproblemas, ambos NP-difíceis de forma independente:

| Subproblema | Descrição | Denominação |
|-------------|----------|-------------|
| *Routing* | Decidir qual máquina executa cada operação | Atribuição |
| *Scheduling* | Decidir a ordem das operações em cada máquina | Sequenciamento |

O número de soluções candidatas cresce de forma super-exponencial com o tamanho do problema. Para as instâncias utilizadas neste trabalho:

| Instância | $n$ | $m$ | Operações | Flex. média | Espaço de busca (ordem de magnitude) |
|-----------|-----|-----|-----------|-------------|--------------------------------------|
| Mk1 | 10 | 6 | 55 | 2.09 | $\sim 10^{57}$ |
| Mk2 | 10 | 6 | 58 | 4.10 | $\sim 10^{60}$ |
| Mk3 | 15 | 8 | 150 | 3.01 | $\sim 10^{130}$ |

Para efeito de comparação, o número estimado de átomos no universo observável é $\sim 10^{80}$. Essa dimensionalidade torna inviável a busca exaustiva e justifica o uso de metaheurísticas como os **Algoritmos Genéticos**.

> **Observação sobre a variante estática:** No FJSP estático, o conjunto completo de jobs, suas operações e os respectivos tempos de processamento são conhecidos antes do início do escalonamento. Isso contrasta com o FJSP dinâmico, no qual novos jobs chegam ao sistema ao longo do tempo, exigindo reescalonamento adaptativo. Neste trabalho, tratamos exclusivamente da variante **estática**.

---
## 3. Referência Bibliográfica Base

A implementação e a metodologia adotadas neste trabalho são inspiradas no seguinte artigo:

> **[1]** *A Genetic Algorithm for the Flexible Job-Shop Scheduling Problem.*  
> ACM Digital Library, 2024.  
> DOI: [10.1145/3698300.3698316](https://dl.acm.org/doi/10.1145/3698300.3698316)

O artigo propõe um Algoritmo Genético com **representação cromossômica dupla** para lidar simultaneamente com os dois subproblemas do FJSP (atribuição e sequenciamento). Os principais aspectos abordados incluem:

- **Codificação dos indivíduos**: cromossomo bipartido, em que uma parte codifica a atribuição de máquinas (*machine assignment*) e outra codifica a ordem de execução das operações (*operation sequence*);
- **Inicialização da população**: uso de heurísticas construtivas (como atribuição à máquina de menor tempo de processamento) combinadas com indivíduos aleatórios;
- **Operadores genéticos**: operadores de cruzamento e mutação adaptados à estrutura dupla do cromossomo;
- **Avaliação experimental**: benchmarks de Brandimarte (Mk1 a Mk10), com comparação frente a outros algoritmos da literatura.

O artigo é utilizado neste trabalho como **referência de comparação**: os resultados obtidos pelo algoritmo implementado serão avaliados frente aos valores reportados no artigo para as mesmas instâncias de Brandimarte. A implementação do Algoritmo Genético em si é de autoria própria, desenvolvida de forma independente sem embasamento direto na implementação descrita no artigo.

---
## 4. Instâncias Utilizadas (Brandimarte)

### 4.1 O Conjunto de Benchmarks de Brandimarte

As instâncias utilizadas neste trabalho pertencem ao conjunto de benchmarks proposto por **Brandimarte (1993)** [[2]](#9-referências), que se tornou uma das referências mais utilizadas na literatura do FJSP. O conjunto é composto por 10 instâncias (Mk1 a Mk10), caracterizadas por diferentes tamanhos e graus de flexibilidade.

Neste trabalho, utilizamos as três primeiras instâncias:

| Instância | Jobs ($n$) | Máquinas ($m$) | Operações | Flex. Média | Melhor Makespan Conhecido |
|-----------|-----------|---------------|-----------|-------------|---------------------------|
| **Mk1** | 10 | 6 | 55 | 2.09 | 40 |
| **Mk2** | 10 | 6 | 58 | 4.10 | 26 |
| **Mk3** | 15 | 8 | 150 | 3.01 | 204 |

A **flexibilidade média** indica, em média, quantas máquinas são elegíveis por operação. Quanto maior esse valor, mais amplo o subproblema de atribuição e maior a dificuldade combinatória.

### 4.2 Formato do Arquivo `.fjs`

As instâncias são armazenadas no formato padrão `.fjs`. Sua estrutura é:

```
Linha 1 :  <nJobs>  <nMáquinas>  [<flexibilidadeMedia>]

Linhas seguintes (uma por job):
  <nOps>  <nMaq_1> <maq> <tempo> ... <nMaq_2> <maq> <tempo> ...  ...
  └──┬──┘  └────────────┬────────┘   └────────────┬────────┘
  nº ops    operação 1              operação 2
```

Cada operação é descrita por: o número de máquinas elegíveis, seguido dos pares `(índice_máquina, tempo_processamento)`.

### 4.3 Exemplo Real — BrandimarteMk1.fjs, Job 1

O arquivo `BrandimarteMk1.fjs` começa com:

```
10  6  2
 6  2 1 5 3 4  3 5 3 3 5 2 1  2 3 4 6 2  3 6 5 2 6 1 1  1 3 1  3 6 6 3 6 4 3
```

**Cabeçalho:** 10 jobs, 6 máquinas, flexibilidade média = 2.

**Job 1** possui `6` operações. Decodificando cada operação:

| Operação | Dado bruto | Máquinas elegíveis ($M_{1,j}$) | Tempos ($p_{1,j,k}$) |
|----------|-----------|-------------------------------|----------------------|
| $O_{1,1}$ | `2  1 5  3 4` | $\{M_1, M_3\}$ | $p_{1,1,1}=5,\ p_{1,1,3}=4$ |
| $O_{1,2}$ | `3  5 3  3 5  2 1` | $\{M_5, M_3, M_2\}$ | $p_{1,2,5}=3,\ p_{1,2,3}=5,\ p_{1,2,2}=1$ |
| $O_{1,3}$ | `2  3 4  6 2` | $\{M_3, M_6\}$ | $p_{1,3,3}=4,\ p_{1,3,6}=2$ |
| $O_{1,4}$ | `3  6 5  2 6  1 1` | $\{M_6, M_2, M_1\}$ | $p_{1,4,6}=5,\ p_{1,4,2}=6,\ p_{1,4,1}=1$ |
| $O_{1,5}$ | `1  3 1` | $\{M_3\}$ | $p_{1,5,3}=1$ |
| $O_{1,6}$ | `3  6 6  3 6  4 3` | $\{M_6, M_3, M_4\}$ | $p_{1,6,6}=6,\ p_{1,6,3}=6,\ p_{1,6,4}=3$ |

Isso evidencia o caráter **FJSP** da instância: 5 das 6 operações do Job 1 possuem 2 ou 3 máquinas elegíveis. A atribuição de máquinas, portanto, não é determinada pelo problema; ela é uma **variável de decisão** que o algoritmo deve otimizar.

Apenas $O_{1,5}$ tem máquina fixa ($M_3$), comportando-se como uma operação de JSSP clássico. Esse misto de operações fixas e flexíveis é típico do **FJSP Parcial** (*Partial Flexibility*), categoria à qual pertencem as instâncias de Brandimarte.

#### Tabela de Resultados do Artigo de Referência

A figura a seguir apresenta os resultados reportados pelo artigo base, comparando o desempenho do AG proposto com outros métodos da literatura nas instâncias de Brandimarte:

<img src="imgs/paper_table2.png" width="600"/>

*Fonte: extraída do artigo de referência [[1]](#9-referências)*

---
## 5. Modelagem para o Algoritmo Genético

Esta seção apresenta a modelagem completa do problema para o AG, a implementação está organizada em quatro etapas:

1. **Leitura da instância** — carregar e estruturar os dados do arquivo `.fjs`
2. **Representação cromossômica** — como um indivíduo é codificado
3. **Decodificação e avaliação** — como um cromossomo é convertido em um escalonamento e avaliado
4. **Inicialização da população** — como gerar a população inicial do AG

### 5.1 Leitura da Instância

Antes de qualquer representação, é necessário carregar os dados do problema na memória. A estrutura central é a lista de operações de cada job, onde cada operação armazena os pares `(máquina, tempo)` das suas opções elegíveis.

Adotamos a seguinte convenção interna:
- Os índices de jobs e máquinas são **0-based** (de 0 a $n-1$ e de 0 a $m-1$).
- `jobs[i][j]` é um dicionário `{máquina: tempo}` representando a operação $O_{i+1, j+1}$.
- `all_operations` é uma lista achatada de todas as operações na ordem `(job_0_op_0, job_0_op_1, ..., job_1_op_0, ...)`, usada para indexar os cromossomos.

In [136]:
import random
from dataclasses import dataclass, field
from typing import Dict, List


@dataclass
class Operation:
    """A single operation belonging to a job.

    Attributes
    ----------
    job_id : int
        0-based index of the job this operation belongs to.
    op_id : int
        0-based position of this operation within the job.
    eligible : Dict[int, int]
        Mapping {machine_id: processing_time} for all eligible machines.
    """
    job_id: int
    op_id: int
    eligible: Dict[int, int]  # {machine_id: processing_time}


@dataclass
class FJSPInstance:
    """Holds all data for a static FJSP instance.

    Attributes
    ----------
    n_jobs : int
        Total number of jobs.
    n_machines : int
        Total number of machines.
    jobs : List[List[Operation]]
        jobs[i] = ordered list of operations for job i.
    all_ops : List[Operation]
        Flat list of all operations in job-major order,
        used as the canonical index space for chromosomes.
    """
    n_jobs: int
    n_machines: int
    jobs: List[List[Operation]]
    all_ops: List[Operation] = field(init=False)

    def __post_init__(self):
        self.all_ops = [op for job in self.jobs for op in job]

    @property
    def n_ops(self) -> int:
        return len(self.all_ops)
    

def load_fjs(path: str) -> FJSPInstance:
    """Parse a .fjs benchmark file and return an FJSPInstance.
    """
    with open(path) as f:
        tokens = f.read().split()

    idx = 0
    n_jobs = int(tokens[idx]);     idx += 1
    n_machines = int(tokens[idx]); idx += 1

    try:
        float(tokens[idx])
        idx += 1 
    except ValueError:
        pass 

    jobs: List[List[Operation]] = []
    for job_id in range(n_jobs):
        n_ops = int(tokens[idx]); idx += 1
        ops: List[Operation] = []
        for op_id in range(n_ops):
            n_eligible = int(tokens[idx]); idx += 1
            eligible: Dict[int, int] = {}
            for _ in range(n_eligible):
                machine = int(tokens[idx]) - 1; idx += 1  # convert to 0-based
                time    = int(tokens[idx]);     idx += 1
                eligible[machine] = time
            ops.append(Operation(job_id=job_id, op_id=op_id, eligible=eligible))
        jobs.append(ops)

    return FJSPInstance(n_jobs=n_jobs, n_machines=n_machines, jobs=jobs)


sample_instance = load_fjs("instancias/BrandimarteMk1.fjs")

print("File loaded: instancias/BrandimarteMk1.fjs")
print(f"Instance loaded: {sample_instance.n_jobs} jobs, {sample_instance.n_machines} machines, {sample_instance.n_ops} total operations")
print()
print("Job 0 — all operations:")
for op in sample_instance.jobs[0]:
    print(f"    Operation {op.op_id} [O({op.job_id},{op.op_id})] - eligible machines: {op.eligible}")

File loaded: instancias/BrandimarteMk1.fjs
Instance loaded: 10 jobs, 6 machines, 55 total operations

Job 0 — all operations:
    Operation 0 [O(0,0)] - eligible machines: {0: 5, 2: 4}
    Operation 1 [O(0,1)] - eligible machines: {4: 3, 2: 5, 1: 1}
    Operation 2 [O(0,2)] - eligible machines: {2: 4, 5: 2}
    Operation 3 [O(0,3)] - eligible machines: {5: 5, 1: 6, 0: 1}
    Operation 4 [O(0,4)] - eligible machines: {2: 1}
    Operation 5 [O(0,5)] - eligible machines: {5: 6, 2: 6, 3: 3}


### 5.2 Representação Cromossômica (Codificação dos Indivíduos)

A principal decisão de modelagem em um AG é como representar uma solução como um cromossomo. Para o FJSP, existem dois subproblemas acoplados que precisam ser codificados simultaneamente:

1. **Para qual máquina cada operação vai?** (subproblema de atribuição)
2. **Em qual ordem as operações são processadas em cada máquina?** (subproblema de sequenciamento)

Adotamos a **codificação dupla** amplamente utilizada na literatura [[3,4]](#9-referências), que divide o cromossomo em dois vetores independentes:

---

#### Vetor 1 — Machine Selection (MS)

Um vetor de comprimento igual ao número total de operações $N_{ops}$. A posição $k$ armazena o **índice da máquina** atribuída à $k$-ésima operação (na ordem achatada `all_ops`).

*Exemplo (Mk1, 55 operações):*
```
MS = [2, 4, 2, 0, 2, 5, ...]
```
Isso traduz-se em:
- 1ª operação (O_{0,0}) → escolha da máquina M_2
- 2ª operação (O_{0,1}) → escolha da máquina M_4
- 3ª operação (O_{0,2}) → escolha da máquina M_2
- 4ª operação (O_{0,3}) → escolha da máquina M_0
- 5ª operação (O_{0,4}) → escolha da máquina M_2
- 6ª operação (O_{0,5}) → escolha da máquina M_5
- ...

---

#### Vetor 2 — Operation Sequence (OS)

Uma **permutação com repetição** de comprimento $N_{ops}$, onde cada job $i$ aparece exatamente $n_i$ vezes. A $k$-ésima ocorrência do valor $i$ representa a $k$-ésima operação do job $i$.

Essa representação é sempre válida (nunca viola a restrição de precedência), pois as operações de cada job são interpretadas sequencialmente pela ordem de aparecimento no vetor.

*Exemplo (3 jobs com 2 operações cada):*
```
OS = [0, 1, 2, 0, 2, 1]
```
Isso traduz-se em:
- 1ª operação do Job 0 (O_{0,0})
- 1ª operação do Job 1 (O_{1,0})
- 1ª operação do Job 2 (O_{2,0})
- 2ª operação do Job 0 (O_{0,1})
- 2ª operação do Job 2 (O_{2,1})
- 2ª operação do Job 1 (O_{1,1})

O vetor OS é então lido da esquerda para a direita para determinar a ordem de escalonamento.

---

Um **indivíduo** é, portanto, o par `(MS, OS)`.

In [137]:
@dataclass
class Chromosome:
    """A candidate solution encoded as a dual chromosome.

    Attributes
    ----------
    ms : List[int]
        Machine Selection vector of length n_ops.
        ms[k] = 0-based index of the machine assigned to all_ops[k].
    os : List[int]
        Operation Sequence vector of length n_ops.
        Contains each job index repeated n_i times.
        The k-th occurrence of job i represents the k-th operation of job i.
    fitness : float
        Makespan of the decoded schedule (lower is better).
    """
    ms: List[int]
    os: List[int]
    fitness: float = None

    def __repr__(self) -> str:
        return f"Chromosome(fitness={self.fitness}, ms={self.ms[:6]}..., os={self.os[:6]}...)"

### 5.3 Decodificação do Cromossomo

A decodificação transforma o par `(MS, OS)` em um escalonamento concreto,ou seja, atribui a cada operação um instante de início $S_{i,j}$ e então calcula o makespan resultante.

O algoritmo percorre o vetor **OS** da esquerda para a direita. A cada passo:

1. Identifica qual operação deve ser escalonada: a $k$-ésima ocorrência do job $i$ no OS corresponde à operação $O_{i,k}$.
2. Consulta o vetor **MS** para saber em qual máquina ela será executada e qual é o tempo de processamento.
3. Calcula o **instante mais cedo possível** para iniciar essa operação, que é o máximo entre:
   - o instante em que a operação anterior do mesmo job terminou (restrição de **precedência**), e
   - o instante em que a máquina ficará disponível (restrição de **capacidade**).
4. Agenda a operação nesse instante e atualiza os contadores de disponibilidade do job e da máquina.

Ao final, o makespan é o maior tempo de término entre todas as operações.

In [138]:
def decode(chromosome: Chromosome, instance: FJSPInstance) -> float:
    """Decode a chromosome into a schedule and return the makespan.

    The decoding follows the active schedule generation procedure:
    operations are scheduled in the order given by the OS vector,
    each starting at the earliest feasible time.

    Parameters
    ----------
    chromosome : Chromosome
        The individual to evaluate.
    instance : FJSPInstance
        The problem data.

    Returns
    -------
    float
        The makespan C_max of the resulting schedule.
    """
    # job_ready[i]     = earliest time job i can start its next operation
    # machine_ready[k] = earliest time machine k is free
    job_ready     = [0] * instance.n_jobs
    machine_ready = [0] * instance.n_machines

    # op_counter[i] = index of the next unscheduled operation of job i
    op_counter = [0] * instance.n_jobs

    # Flat index into all_ops for each job, used to look up MS
    # job_op_flat_index[i][j] = flat index of operation j of job i
    job_op_flat_index: List[List[int]] = []
    flat_idx = 0
    for job in instance.jobs:
        indices = []
        for _ in job:
            indices.append(flat_idx)
            flat_idx += 1
        job_op_flat_index.append(indices)

    makespan = 0

    for job_id in chromosome.os:
        op_pos   = op_counter[job_id]           # which operation of this job
        flat_idx = job_op_flat_index[job_id][op_pos]
        op       = instance.all_ops[flat_idx]
        machine  = chromosome.ms[flat_idx]       # assigned machine (0-based)
        proc_time = op.eligible[machine]

        # Earliest start: respect both job precedence and machine availability
        start = max(job_ready[job_id], machine_ready[machine])
        end   = start + proc_time

        # Update readiness
        job_ready[job_id]     = end
        machine_ready[machine] = end
        op_counter[job_id]    += 1

        if end > makespan:
            makespan = end

    return float(makespan)

### 5.4 Função de Aptidão (*Fitness*)

A função de aptidão quantifica a qualidade de um indivíduo. Como o objetivo do FJSP é **minimizar** o makespan, e os algoritmos genéticos tradicionais **maximizam** a aptidão, adotamos a convenção de usar o makespan diretamente como medida de custo (quanto menor, melhor).

Optamos por trabalhar com **minimização direta do makespan** ao longo de todo o AG, evitando transformações desnecessárias que poderiam introduzir distorções na seleção.

A função abaixo avalia um cromossomo, armazena o resultado no campo `fitness` e o retorna:

In [139]:
def evaluate(chromosome: Chromosome, instance: FJSPInstance) -> float:
    """Compute and store the fitness (makespan) of a chromosome.

    Parameters
    ----------
    chromosome : Chromosome
    instance   : FJSPInstance

    Returns
    -------
    float
        Makespan C_max (same value stored in chromosome.fitness).
    """
    chromosome.fitness = decode(chromosome, instance)
    return chromosome.fitness

### 5.5 Inicialização da População

A qualidade da população inicial influencia significativamente a velocidade de convergência do AG. Adotamos uma estratégia **mista**, combinando dois tipos de indivíduos:

#### Estratégia 1 — Inicialização Aleatória

Para cada indivíduo aleatório:
- **MS:** para cada operação, sorteia-se uniformemente uma das máquinas elegíveis.
- **OS:** gera-se uma permutação aleatória da lista `[0]*n0 + [1]*n1 + ... + [n-1]*n_{n-1}`, onde $n_i$ é o número de operações do job $i$.

A aleatoriedade garante **diversidade genética** na população inicial, evitando convergência prematura.

#### Estratégia 2 — Inicialização por Heurística SPT (*Shortest Processing Time*)

Para cada operação, a máquina com o **menor tempo de processamento** é selecionada. Essa heurística construtiva produz indivíduos de qualidade superior à média aleatória, acelerando o início da convergência. O vetor OS é, ainda assim, gerado aleatoriamente para manter diversidade no sequenciamento.

A combinação dos dois tipos mantém o equilíbrio entre **exploração** (diversidade) e **explotação** (qualidade inicial).

In [140]:
def _random_os(instance: FJSPInstance) -> List[int]:
    """Generate a random, valid Operation Sequence (OS) vector.

    The OS is a permutation of the multiset {0 * n0, 1 * n1, ..., (n-1) * n_{n-1}},
    where ni is the number of operations in job i.
    """
    os_list = [job_id for job_id, job in enumerate(instance.jobs) for _ in job]
    random.shuffle(os_list)
    return os_list


def random_chromosome(instance: FJSPInstance) -> Chromosome:
    """Create a fully random chromosome.

    MS: for each operation, pick a machine uniformly at random from eligible set.
    OS: random valid permutation (see _random_os).
    """
    ms = [random.choice(list(op.eligible.keys())) for op in instance.all_ops]
    os = _random_os(instance)
    return Chromosome(ms=ms, os=os)


def spt_chromosome(instance: FJSPInstance) -> Chromosome:
    """Create a chromosome using the Shortest Processing Time (SPT) heuristic.

    MS: for each operation, assign the machine with the minimum processing time.
    OS: random valid permutation (keeps diversity in sequencing).
    """
    ms = [
        min(op.eligible, key=lambda m: op.eligible[m])
        for op in instance.all_ops
    ]
    os = _random_os(instance)
    return Chromosome(ms=ms, os=os)


def init_population(
    instance: FJSPInstance,
    pop_size: int,
    spt_ratio: float = 0.2,
    seed: int | None = None,
) -> List[Chromosome]:
    """Initialise and evaluate the GA population.

    Parameters
    ----------
    instance  : FJSPInstance
    pop_size  : int
        Total number of individuals in the population.
    spt_ratio : float  (default 0.2)
        Fraction of the population generated by the SPT heuristic.
        The remainder (1 - spt_ratio) is generated randomly.
    seed : int or None
        Random seed for reproducibility.

    Returns
    -------
    List[Chromosome]
        Evaluated population sorted by fitness (ascending = better first).
    """
    if seed is not None:
        random.seed(seed)

    n_spt    = max(1, int(pop_size * spt_ratio))
    n_random = pop_size - n_spt

    population: List[Chromosome] = []

    # Heuristic individuals (SPT)
    for _ in range(n_spt):
        ind = spt_chromosome(instance)
        evaluate(ind, instance)
        population.append(ind)

    # Random individuals
    for _ in range(n_random):
        ind = random_chromosome(instance)
        evaluate(ind, instance)
        population.append(ind)

    population.sort(key=lambda c: c.fitness)
    return population

### 5.6 Verificação da Modelagem

Antes de prosseguir para a implementação do AG, vale validar as componentes desenvolvidas acima com a instância `BrandimarteMk1`. Esperamos que:

- O cromossomo SPT produzirá um makespan inferior ao randômico médio;
- O melhor indivíduo da população inicial ficará bem acima do makespan ótimo conhecido (40), já que não há nenhuma otimização ainda — apenas inicialização;
- A estrutura interna do cromossomo estará correta (tamanho, domínio dos valores).

In [141]:
SEED = None
POP_SIZE = 50
SPT_RATIO = 0.3

population = init_population(instance, pop_size=POP_SIZE, spt_ratio=SPT_RATIO, seed=SEED)

fitnesses = [ind.fitness for ind in population]
best      = population[0]
worst     = population[-1]

print(f"  Instance  : BrandimarteMk1")
print(f"  Pop. size : {POP_SIZE}  |  SPT ratio: {SPT_RATIO*100:.0f}%")
print(f"  Seed      : {SEED}")
print()
print("=" * 55)
print(f"  Best  makespan (initial pop.) : {best.fitness:.0f}")
print(f"  Worst makespan (initial pop.) : {worst.fitness:.0f}")
print(f"  Mean  makespan                : {sum(fitnesses)/len(fitnesses):.1f}")
print(f"  Known optimum                 : 40")
print()
print("=" * 55)
print("Best chromosome structure:")
print(f"  MS length : {len(best.ms)}")
print(f"  OS length : {len(best.os)}")
print(f"  MS[:8]    : {best.ms[:10]}")
print(f"  OS[:8]    : {best.os[:10]}")
print()

# Verify that every MS value is within the eligible set
violations = sum(
    1 for k, op in enumerate(instance.all_ops)
    if best.ms[k] not in op.eligible
)
print(f"  MS constraint violations : {violations}")

# Verify that every job appears n_i times in OS
from collections import Counter
os_count = Counter(best.os)
seq_ok = all(os_count[j] == len(instance.jobs[j]) for j in range(instance.n_jobs))
print(f"  OS job-count valid       : {seq_ok}")

  Instance  : BrandimarteMk1
  Pop. size : 50  |  SPT ratio: 30%
  Seed      : None

  Best  makespan (initial pop.) : 70
  Worst makespan (initial pop.) : 113
  Mean  makespan                : 87.8
  Known optimum                 : 40

Best chromosome structure:
  MS length : 55
  OS length : 55
  MS[:8]    : [2, 1, 5, 0, 2, 3, 1, 2, 0, 1]
  OS[:8]    : [0, 5, 2, 6, 4, 1, 8, 2, 3, 4]

  MS constraint violations : 0
  OS job-count valid       : True


### 5.7 Resumo da Modelagem

A tabela abaixo resume as escolhas de modelagem feitas nesta seção:

| Componente | Escolha | Justificativa |
|------------|---------|---------------|
| **Representação** | Cromossomo duplo (MS + OS) | Separa naturalmente os dois subproblemas. Facilitando a criação de operadores específicos para cada parte |
| **MS** | Índice de máquina por operação | Mapeamento direto e compacto. Permitindo cruzamento simples por ponto de corte |
| **OS** | Permutação com repetição de job-ids | Garante validade de precedência por construção. Onde qualquer permutação é decodificável |
| **Decodificador** | Escalonamento ativo | Produz soluções viáveis. Escalonamentos ativos contêm o ótimo |
| **Fitness** | Makespan (minimizar) | Métrica padrão da literatura. Facilita comparação com benchmarks |
| **Inicialização** | 30% SPT + 70% aleatório | Equilíbrio entre qualidade inicial e diversidade genética |

---
## 6. Implementação

> 🔧 *Esta seção será desenvolvida na próxima etapa do trabalho.*

Tópicos a cobrir:
- Seleção (torneio, roleta)
- Cruzamento (*crossover*)
- Mutação
- Parâmetros do AG (tamanho da população, taxa de cruzamento, taxa de mutação, critério de parada)
- Código-fonte

---
## 7. Experimentos e Resultados

> 🔧 *Esta seção será desenvolvida na próxima etapa do trabalho.*

Tópicos a cobrir:
- Configuração experimental (hardware, repetições, sementes)
- Resultados por instância (Mk1, Mk2, Mk3): melhor, médio, pior makespan
- Comparação com os valores do artigo de referência
- Curva de convergência do AG

---
## 8. Conclusão

> 🔧 *Esta seção será desenvolvida ao final do trabalho.*

---
## 9. Referências

**[1]** *A Genetic Algorithm for the Flexible Job-Shop Scheduling Problem.* ACM Digital Library, 2024.  
DOI: [10.1145/3698300.3698316](https://dl.acm.org/doi/10.1145/3698300.3698316)

**[2]** Brandimarte, P. (1993). Routing and scheduling in a flexible job shop by tabu search. *Annals of Operations Research*, 41(3), 157–183.

**[3]** Kacem, I., Hammadi, S., & Borne, P. (2002). Approach by localization and multiobjective evolutionary optimization for flexible job-shop scheduling problems. *IEEE Transactions on Systems, Man, and Cybernetics*, 32(1), 408–419.

**[4]** Xia, W., & Wu, Z. (2005). An effective hybrid optimization approach for multi-objective flexible job-shop scheduling problems. *Computers & Industrial Engineering*, 48(2), 409–425.

**[5]** Garey, M. R., & Johnson, D. S. (1979). *Computers and Intractability: A Guide to the Theory of NP-Completeness.* W. H. Freeman.